# Small Language Model

Transformer network to solve simple maths questions.

## Setup

In [ ]:
# @title Imports
%pip install lightning --quiet

import os
import torch
import pytorch_lightning as pl

from llm_pet.data_set.gsm8k.gsm8k_access import get_gsm8k_rows
from llm_pet.data_set.gsm8k.gsm8k_lightning import Gsm8kDataModule
from llm_pet.data_set.tiny_stories.tiny_stories_lightning import TinyStoriesDataModule
from llm_pet.data_set.vocab import VOCAB_SIZE, encode, decode, BOS_ID, EOS_ID
from llm_pet.training.callbacks import ShardSwapCallback, GraceEarlyStopping
from llm_pet.training.rl.opt_step import RL_NUM_QUESTIONS, rl_finetune
from llm_pet.training.rl.reward import compute_reward
from llm_pet.training.slm_lightning import SLMLightning

In [ ]:
# @title determine device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [ ]:
# @title check data set exists
assert os.path.isfile('train-00000-of-00001.parquet'), "Can't fine train set!"
assert os.path.isfile('test-00000-of-00001.parquet'), "Can't fine test set!"

## Model Specification

In [ ]:
# hyper parameters
NUM_LAYERS = 6
DIM = 384
NUM_HEADS = 6
MAX_CONTEXT = 128
P_DROP = 0.2

## Initial Training

Train the model on the English language, using the tiny stories dataset.

In [ ]:
from llm_pet.data_set.tiny_stories.tiny_stories_access import ROWS_PER_SHARD, VALIDATION_ROWS

BATCH_SIZE = 64
LEARNING_RATE = 3e-4
VAL_CHECK_INTERVAL = 1
MAX_STEPS = 1

pl.seed_everything(42, workers=True)

datamodule = TinyStoriesDataModule(
    rows_per_shard=ROWS_PER_SHARD, validation_rows=VALIDATION_ROWS,
    batch_size=BATCH_SIZE, max_context=MAX_CONTEXT, seed=42)

model = SLMLightning(
    num_layers=NUM_LAYERS, dim=DIM, num_heads=NUM_HEADS, max_context=MAX_CONTEXT,
    vocab_size=VOCAB_SIZE, p_drop=P_DROP, learning_rate=LEARNING_RATE,
    max_steps=MAX_STEPS)

swap_cb = ShardSwapCallback(datamodule, abs_gap=0.30, ratio=0.80)
early_stop_cb = GraceEarlyStopping(
    monitor="val_loss", mode="min", patience=10, min_delta=1e-3,
    grace_period=5, verbose=True)
swap_cb.early_stopping = early_stop_cb

trainer = pl.Trainer(
    max_steps=MAX_STEPS,
    val_check_interval=VAL_CHECK_INTERVAL,
    check_val_every_n_epoch=None,
    # swap_cb MUST come before early_stop_cb so a swap sets the grace period
    # before that same validation round's stopping check runs.
    callbacks=[swap_cb, early_stop_cb],
    accelerator="auto",
    devices="auto",
    log_every_n_steps=25,
    gradient_clip_val=1.0,
    reload_dataloaders_every_n_epochs=1,
)

trainer.fit(model, datamodule=datamodule)

# a trained model can be loaded form a checkpoint like this:
# model = SLMLightning.load_from_checkpoint("path/to/checkpoint.ckpt")
# mdoel.eval()

In [ ]:
def generate(context_text):
  model.eval()
  ctx_ids = encode(context_text)[-model.hparams.max_context:]
  context = torch.tensor([ctx_ids], device=model.device)
  out = torch.tensor([[BOS_ID]], device=model.device)

  generated = model.model.generate(context, out, max_tokens=100)[0].tolist()
  if EOS_ID in generated:
      generated = generated[: generated.index(EOS_ID)]
  return decode(generated)

for ctx, dat_set in [
    ("Once upon a time, there was a little car named Beep.", "Training"),
    ("One day, a young boy named Tim found a dull, round rock.", "Validation"),
    ("In 1950, a young doctor named Norton Perina embarks ona an expedition.", "Real"),
]:
    print(f"context : {ctx}")
    print(f"next    : {generate(ctx)}\n")

## Supervised Fine Tuning

In [ ]:
FT_BATCH_SIZE = 128
FT_LEARNING_RATE = 1e-5          # gentler LR for fine-tuning a pretrained model
FT_MAX_STEPS = 5000
FT_VAL_CHECK_INTERVAL = 50

pl.seed_everything(42, workers=True)

# Start from the pretrained TinyStories checkpoint loaded earlier in `model`.
# We keep its weights but re-point the optimizer/scheduler at the fine-tuning
# schedule (and lower LR) via the hparams below. max_context stays 128 to match
# the pretrained position embeddings.
model.learning_rate = FT_LEARNING_RATE
model.max_steps_total = FT_MAX_STEPS

gsm8k_datamodule = Gsm8kDataModule(
    rows_per_shard=ROWS_PER_SHARD, validation_rows=VALIDATION_ROWS,
    batch_size=FT_BATCH_SIZE, max_context=MAX_CONTEXT, seed=42)

ft_swap_cb = ShardSwapCallback(gsm8k_datamodule, abs_gap=0.30, ratio=0.80)
ft_early_stop_cb = GraceEarlyStopping(
    monitor="val_loss", mode="min", patience=10, min_delta=1e-3,
    grace_period=5, verbose=True)
ft_swap_cb.early_stopping = ft_early_stop_cb     # let swaps trigger the grace period

ft_trainer = pl.Trainer(
    max_steps=FT_MAX_STEPS,
    val_check_interval=FT_VAL_CHECK_INTERVAL,
    check_val_every_n_epoch=None,                # validate by step count
    # swap_cb MUST come before early_stop_cb so a swap sets the grace period
    # before that same validation round's stopping check runs.
    callbacks=[ft_swap_cb, ft_early_stop_cb],
    accelerator="auto",
    devices="auto",
    log_every_n_steps=25,
    gradient_clip_val=1.0,
    reload_dataloaders_every_n_epochs=1,         # honour mid-run shard swaps
)

ft_trainer.fit(model, datamodule=gsm8k_datamodule)

In [ ]:

def answer_question(question_text, max_tokens=None):
    """Feed a GSM8k question as context and let the model generate the answer.

    max_tokens is capped at max_context (128) because the decoder's fixed
    position embeddings can't address positions beyond that.
    """
    model.eval()
    max_tokens = max_tokens or model.hparams.max_context   # <= 128
    ctx_ids = encode(question_text)[-model.hparams.max_context:]
    context = torch.tensor([ctx_ids], device=model.device)
    out = torch.tensor([[BOS_ID]], device=model.device)

    generated = model.model.generate(context, out, max_tokens=max_tokens)[0].tolist()
    generated = generated[1:]                    # drop the leading <bos>
    if EOS_ID in generated:
        generated = generated[: generated.index(EOS_ID)]
    return decode(generated)


for q in [
    "Natalia sold clips to 48 of her friends in April, and then she sold half "
    "as many clips in May. How many clips did Natalia sell altogether in April "
    "and May?",
    "Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes "
    "of babysitting. How much did she earn?",
    "Betty is saving money for a new wallet which costs $100. Betty has only "
    "half of the money she needs. Her parents decided to give her $15 for that "
    "purpose, and her grandparents twice as much as her parents. How much more "
    "money does Betty need to buy the wallet?",
]:
    print(f"question : {q}")
    print(f"answer   : {answer_question(q)}\n")

## RL Fine Tuning

In [ ]:
pl.seed_everything(42, workers=True)

rl_rows, _ = get_gsm8k_rows("train", 0, RL_NUM_QUESTIONS, MAX_CONTEXT)
print(f"[rl] loaded {len(rl_rows)} GSM8k (question, answer) pairs for RL")

# sanity-check the reward on the gold answers before training: a gold answer
# should score highly against itself (full presence + match + final*2.0).
_q0, _a0 = rl_rows[0]
print(f"[rl] sample gold reward (self-score) = {compute_reward(_a0, _a0):.3f}")

model = rl_finetune(model, rl_rows)

# @title Compare answers before/after is best done by re-running the GSM8k
# sampling cell above; here we just show a few post-RL generations.
for q in [
    "Natalia sold clips to 48 of her friends in April, and then she sold half "
    "as many clips in May. How many clips did Natalia sell altogether in April "
    "and May?",
    "Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes "
    "of babysitting. How much did she earn?",
]:
    print(f"question : {q}")
    print(f"answer   : {answer_question(q)}\n")